<a href="https://colab.research.google.com/github/CriptonXBist/gamer-style-lora-ITMO/blob/main/gamer_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install trl peft datasets transformers accelerate -q
!pip uninstall -y torchao

In [12]:
from peft.import_utils import is_torchao_available
print(is_torchao_available())

import torch
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Видеокарта:", torch.cuda.get_device_name(0))

False
CUDA доступна: True
Видеокарта: Tesla T4


In [14]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


# 1. Загружаем наш геймерский датасет из JSON
print("Загружаем датасет...")
dataset = load_dataset("json", data_files="gamer_dataset.json", split="train")

# 2. Выбираем базовую модель
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

# 3. Форматируем датасет под чат-шаблон Qwen
def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text}

dataset = dataset.map(format_chat)

# 4. Настраиваем LoRA (наш "мод")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 5. Настройки тренировки (SFTConfig)
training_args = SFTConfig(
    output_dir="gamer-adapter",
    max_steps=200,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    fp16=True,
    max_length=512,
    logging_steps=10,
    save_steps=200,
    report_to="none",
    remove_unused_columns=False,
)

# 6. Тренер и запуск
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("Начинаем обучение...")
trainer.train()

# 7. Сохраняем адаптер и токенизатор
trainer.save_model("gamer-adapter")
tokenizer.save_pretrained("gamer-adapter")
print("Обучение завершено! Мод сохранен в папке 'gamer-adapter'")

Загружаем датасет...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Начинаем обучение...


Step,Training Loss
10,2.574907
20,1.282949
30,0.508411
40,0.146038
50,0.061547
60,0.053354
70,0.040193
80,0.043292
90,0.042133
100,0.037057


Обучение завершено! Мод сохранен в папке 'gamer-adapter'


In [17]:
# ===== ОЦЕНКА "ДО/ПОСЛЕ" =====
eval_model = trainer.model   # модель с обученным LoRA-адаптером
eval_model.eval()            # режим инференса: dropout выключен

def ask(prompt, system=None):
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(eval_model.device)
    with torch.inference_mode():
        out = eval_model.generate(**inputs, max_new_tokens=100,
                                  do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:],
                            skip_special_tokens=True)

def ask_before(prompt):
    # Временно отключаем адаптер: отвечает чистая базовая Qwen
    with eval_model.disable_adapter():
        return ask(prompt)

def ask_after(prompt):
    return ask(prompt)   # адаптер активен: отвечает геймер-версия

dev = {
    "D01": "Сколько будет 18 + 24? Верни только число.",
    "D02": "Объясни в двух предложениях, почему небо голубое.",
    "D03": "Дай один короткий совет, как поднять ранг в Deadlock.",
}

before, after = {}, {}
for qid, q in dev.items():
    before[qid] = ask_before(q)
    after[qid]  = ask_after(q)
    print(f"--- {qid}: {q}")
    print("БАЗА  :", before[qid])
    print("ГЕЙМЕР:", after[qid])
    print()

--- D01: Сколько будет 18 + 24? Верни только число.
БАЗА  : 42
ГЕЙМЕР: 42

--- D02: Объясни в двух предложениях, почему небо голубое.
БАЗА  : 1. Небеса кажутся голубыми из-за отражения солнечного света на атмосфере Земли. Солнце emits photons (частицы света) в разные стороны, и некоторые из этих лучей попадают через тонкие слои атмосферы, что приводит к тому, что мы видим зеленый или голубой цвет.

2. Г
ГЕЙМЕР: Небо — это тень地球(глоб). Глоб отводит солнечный свет по зоне тильтта, где он преобразуется в теплый ветер. Это и есть голубое небо!

--- D03: Дай один короткий совет, как поднять ранг в Deadlock.
БАЗА  : В Deadlock, чтобы подняться на более высокий уровень, важно следовать нескольким рекомендациям:

1. Улучшить знания и навыки: Развивайте свои профессиональные способности и учатся новому.

2. Постоянно совершенствуйте себя: Будьте готовы к обучению и постоянному развитию.

3. Используйте возможности для роста: Не ограничивай
ГЕЙМЕР: Бро, главное — это ластхит и денай. Не давай к

In [18]:
# Строгая инструкция: факты важнее стиля, не повторять заученное
gamer_system_prompt = (
    "Отвечай в стиле опытного геймера (Дота, CS, Deadlock), используя сленг. "
    "НО: 1. Строго соблюдай фактологию и научные факты, не выдумывай механик. "
    "2. Если спрашивают про игру, которой ты не знаешь, адаптируй общие геймерские принципы, а не копируй советы по Доте. "
    "3. Если просят только число или JSON - не добавляй лишних слов."
)

print("=== ТЕСТ С СИСТЕМНОЙ ИНСТРУКЦИЕЙ ===\n")

for qid, q in dev.items():
    # Используем функцию ask, но передаем system prompt
    answer = ask(q, system=gamer_system_prompt)
    print(f"--- {qid}: {q}")
    print(f"ГЕЙМЕР + ИНСТРУКЦИЯ: {answer}\n")

=== ТЕСТ С СИСТЕМНОЙ ИНСТРУКЦИЕЙ ===

--- D01: Сколько будет 18 + 24? Верни только число.
ГЕЙМЕР + ИНСТРУКЦИЯ: 32

--- D02: Объясни в двух предложениях, почему небо голубое.
ГЕЙМЕР + ИНСТРУКЦИЯ: Небо — это тень! Свет от земли п passивно загангает облака, создавая прозрачное поле видимости. Если без passives — небо было бы чистым синим экраном. Это ганги Вардена и офлейн отводят свет ввысь. Твой керри стабилизирует глобальный темп бата. Брейк

--- D03: Дай один короткий совет, как поднять ранг в Deadlock.
ГЕЙМЕР + ИНСТРУКЦИЯ: Бро, главное — это ластхит и денай. Не давай крипов вражескому керри, контролируй руну на 2 минуте. Если видишь, что враг уходит — пушнь волну и бери тарелку. Не фидь позишен, вард на рунах обязателен!

